In [6]:
import configparser
from os import getcwd
from pytz import timezone
import pandas as pd
from yfpy.query import YahooFantasySportsQuery
from nba_api.stats.library.parameters import Season
from sqlalchemy.sql import text
from datetime import datetime, date, timedelta
from sqlalchemy.dialects.postgresql.base import PGDialect; PGDialect._get_server_version_info = lambda *args: (9, 2)
from dataHub import dataHub
dh = dataHub()

db_con = dh.db_connect('postgre')
fty_con = dh.fty_con(db_con)

fty_con = fty_con['Yahoo;121793']

In [52]:
df = []
for activity in fty_con.get_league_transactions():
    if activity.type != 'commish':
        for player in activity.players:
            el_id = [el for el in player.clean_data_dict()['transaction_data'].keys() if el.endswith('_team_key')][0]
  
            df.append({
                'season': Season.current_season,
                'platform': 'Yahoo',
                'league_id': fty_con.league_id,
                'timestamp': datetime.fromtimestamp(activity.timestamp),
                'competitor_id': int(player.clean_data_dict()['transaction_data'][el_id].replace('454.l.121793.t.', '')),
                'action': player.clean_data_dict()['transaction_data']['type'],
                'player': player.clean_data_dict()['name']['full']
            })
            
df_t = pd.read_sql(f"SELECT * FROM fty.recent_activity WHERE season = '{Season.current_season}' AND platform = 'Yahoo' AND league_id = {fty_con.league_id}", db_con)
df = pd.DataFrame(df).merge(df_t, how='outer', indicator=True)
df = df[(df._merge=='left_only')].drop('_merge', axis=1)
df

2024-10-17 18:58:42.298 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.


,season,platform,league_id,timestamp,competitor_id,action,player,competitor_name
0,2024-25,Yahoo,121793,2024-10-17 17:58:59,5,add,Khris Middleton,NaN
1,2024-25,Yahoo,121793,2024-10-17 17:57:41,5,add,Jaden Ivey,NaN
2,2024-25,Yahoo,121793,2024-10-17 08:38:33,8,add,Keyonte George,NaN
3,2024-25,Yahoo,121793,2024-10-17 08:38:33,8,drop,Chris Paul,NaN
4,2024-25,Yahoo,121793,2024-10-16 20:52:25,1,add,Marcus Smart,NaN
5,2024-25,Yahoo,121793,2024-10-16 20:49:52,1,add,Mike Conley,NaN
6,2024-25,Yahoo,121793,2024-10-16 20:08:55,1,add,Donte DiVincenzo,NaN


In [15]:
# dh.fty_get_league_competitor(fty_con, db_con)
# dh.fty_get_league_schedule(fty_con, db_con)
# dh.fty_get_competitor_roster(fty_con, db_con)
# dh.fty_get_free_agents(fty_con, db_con)
# dh.fty_get_recent_activity(fty_con, db_con)

In [2]:
for team in [team.team_id for team in query.get_league_teams()]:
    query.get_team_stats_by_week(team)

2024-10-16 20:53:46.017 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-16 20:53:47.724 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.


KeyError: 'team_projected_points'